## load the Deep Fashion dataset

In [ ]:
# the relative imports work if the module is a part of a package, and the script is being executed 
# as a module and this doesn't apply on jupyter notebook

# so we need to add the parent folder to sys.path
import sys 
import os 

from pathlib import Path
path = os.path.abspath(Path(os.getcwd()).parent)
print(path)

In [3]:
# import this path to sys.path
sys.path.append(path)

from preprocessing import config as C
import pandas as pd 


### load category images

In [ ]:
category_images = pd.read_csv(
    C.ANNOTATIONS_DIR / "category" /"list_category_img.txt",
    sep = r"\s+",
    header= None,
    skiprows= 2,
    names= ['file', 'category_label']
)

In [ ]:
category_images.head()

In [ ]:
print(category_images["category_label"].unique())

### load categorical cloth

In [ ]:
category_cloth = pd.read_csv(
    C.ANNOTATIONS_DIR / "category" / "list_category_cloth.txt",
    sep = r"\s+",
    skiprows= 2,
    header= None,
    names= ['category_name', 'category_type']
).reset_index(names="category_label") #the category_label is the index of the category_cloth now

# add the region column
category_cloth["region"] = category_cloth["category_type"].map({1: "upper", 2: "lower", 3: "full"})

category_cloth.head()

In [ ]:
category_cloth.shape

### merge the two datasets 


In [ ]:
df = category_images.merge(category_cloth, on="category_label", how="left")

df.head()

### add the bbox coordinate and the split column

In [ ]:

# 4. add the bbox coordinate
bbox_coordinate = pd.read_csv(
    C.ANNOTATIONS_DIR / "bbox" / "list_bbox.txt",
    sep=r"\s+",
    header=None,
    skiprows=2,
    names = ["file", "x_1", "y_1", "x_2", "y_2"]
)

df = df.merge(bbox_coordinate,on="file" ,how="left")

# 5. add the split info
split = pd.read_csv(
    C.ANNOTATIONS_DIR  / "splits" / "list_eval_partition.txt",
    sep=r"\s+", header=None, skiprows=2,
    names=["file", "split"]
)
df = df.merge(split, on="file", how="left")
df.head()

In [ ]:
df["split"].value_counts() / len(df) * 100 

In [ ]:
df.describe()

## load the Fashion Gen dataset 

In [42]:
# load the styles.csv file 

styles = pd.read_csv(C.STYLES_GEN_DIR, on_bad_lines='warn', engine="python")

# rename the id to file
styles = styles.rename(columns={"id":"file", "subCategory" : "category"})

Skipping line 6044: Expected 10 fields in line 6044, saw 11
Skipping line 6569: Expected 10 fields in line 6569, saw 11
Skipping line 7399: Expected 10 fields in line 7399, saw 11
Skipping line 7939: Expected 10 fields in line 7939, saw 11
Skipping line 9026: Expected 10 fields in line 9026, saw 11
Skipping line 10264: Expected 10 fields in line 10264, saw 11
Skipping line 10427: Expected 10 fields in line 10427, saw 11
Skipping line 10905: Expected 10 fields in line 10905, saw 11
Skipping line 11373: Expected 10 fields in line 11373, saw 11
Skipping line 11945: Expected 10 fields in line 11945, saw 11
Skipping line 14112: Expected 10 fields in line 14112, saw 11
Skipping line 14532: Expected 10 fields in line 14532, saw 11
Skipping line 15076: Expected 10 fields in line 15076, saw 12
Skipping line 29906: Expected 10 fields in line 29906, saw 11
Skipping line 31625: Expected 10 fields in line 31625, saw 11
Skipping line 33020: Expected 10 fields in line 33020, saw 11
Skipping line 3574

In [43]:
# missing category rows 
styles["category"].isnull().sum()


0

### add the .jpg extension to file name 

In [44]:
styles["file"] = styles["file"].astype(str) + ".jpg"

In [45]:
styles.head()

,file,gender,masterCategory,category,articleType,baseColour,season,year,usage,productDisplayName
0,15970.jpg,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt
1,39386.jpg,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans
2,59263.jpg,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch
3,21379.jpg,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants
4,53759.jpg,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt


### add a split column

we will use 80-10-10 logic:

    - 0==>7 : train
    - 8: val
    - 9: test

In [52]:
styles["split"] = (
    styles.groupby("category").cumcount()
    .apply(lambda i : "train" if i%10 <8 else ("val" if i%10 == 8 else "test"))
)

styles.head()

,file,gender,masterCategory,category,articleType,baseColour,season,year,usage,productDisplayName,split
0,15970.jpg,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt,train
1,39386.jpg,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans,train
2,59263.jpg,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch,train
3,21379.jpg,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants,train
4,53759.jpg,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt,train


In [59]:
styles["split"].value_counts() * 100 / len(styles)

train    80.085089
val       9.963083
test      9.951828
Name: split, dtype: float64